# TruePrice YOLO fruit detector training

Use this notebook in Google Colab with GPU runtime. Upload `dataset_open_v1/yolo_fruit_only.zip` or place it in Google Drive, then run cells from top to bottom.

In [ ]:
# 1. Check runtime
import os
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Switch Colab runtime to GPU before full training.')

In [ ]:
# 2. Install training dependency
!pip -q install ultralytics

## Dataset input

Recommended MVP path: upload the local zip file below through the next cell.

Local source file:
`/Users/shyoon840/HGU/3-1/HCI/TeamProject/hci_222/dataset_open_v1/yolo_fruit_only.zip`

If browser upload is too slow, upload the zip to Google Drive first and set `DATASET_ZIP` to that Drive path after mounting Drive.

In [ ]:
# 3A. Upload zip directly to Colab.
# Skip this cell if you already copied the zip to /content or mounted Drive.
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

In [ ]:
# 3B. Optional Drive path instead of browser upload.
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ZIP = '/content/drive/MyDrive/trueprice/yolo_fruit_only.zip'

DATASET_ZIP = '/content/yolo_fruit_only.zip'
DATASET_ROOT = '/content/dataset'
YOLO_ROOT = f'{DATASET_ROOT}/yolo'

assert os.path.exists(DATASET_ZIP), f'Missing dataset zip: {DATASET_ZIP}'
print('Dataset zip:', DATASET_ZIP)

In [ ]:
# 4. Unzip and fix absolute local path in data yaml
from pathlib import Path

!rm -rf /content/dataset
!mkdir -p /content/dataset
!unzip -q "$DATASET_ZIP" -d /content/dataset

yaml_path = Path('/content/dataset/yolo/data_fruit_only.yaml')
assert yaml_path.exists(), f'Missing yaml: {yaml_path}'

lines = yaml_path.read_text().splitlines()
lines = [f'path: {YOLO_ROOT}' if line.startswith('path:') else line for line in lines]
yaml_path.write_text('\n'.join(lines) + '\n')
print(yaml_path.read_text())

In [ ]:
# 5. Quick dataset sanity check
from pathlib import Path

for split in ['train', 'valid', 'test']:
    image_count = len(list(Path(YOLO_ROOT, split, 'images').glob('*')))
    label_count = len(list(Path(YOLO_ROOT, split, 'labels').glob('*.txt')))
    print(split, 'images=', image_count, 'labels=', label_count)

In [ ]:
# 6. Smoke test training. Run this first to catch dataset/config errors quickly.
!yolo detect train model=yolov8n.pt data=/content/dataset/yolo/data_fruit_only.yaml imgsz=640 epochs=3 batch=16 workers=2 project=/content/runs name=smoke

In [ ]:
# 7. Full MVP training. Increase epochs only if metrics are still improving.
!yolo detect train model=yolov8n.pt data=/content/dataset/yolo/data_fruit_only.yaml imgsz=640 epochs=50 batch=16 workers=2 patience=10 project=/content/runs name=fruit_yolov8n

In [ ]:
# 8. Export for Flutter/TFLite runtime
BEST_PT = '/content/runs/fruit_yolov8n/weights/best.pt'
assert os.path.exists(BEST_PT), f'Missing trained weight: {BEST_PT}'

!yolo export model="$BEST_PT" format=tflite imgsz=640
!find /content/runs -name '*.tflite' -o -name 'best.pt' -o -name 'results.csv'

In [ ]:
# 9. Download artifacts
from pathlib import Path
from google.colab import files

artifact_paths = []
artifact_paths += list(Path('/content/runs').glob('fruit_yolov8n/weights/best.pt'))
artifact_paths += list(Path('/content/runs').rglob('*.tflite'))
artifact_paths += list(Path('/content/runs').glob('fruit_yolov8n/results.csv'))

for path in artifact_paths:
    print('download:', path)
    files.download(str(path))

## Flutter handoff

Copy the exported `.tflite` model into `frontend/flutter_app/assets/models/`, add it to `pubspec.yaml`, then wire inference with `tflite_flutter` or keep FastAPI inference if server-side inference is faster for the MVP.